# 04. Analytics e Insights — Copa do Mundo 2026

Neste notebook, realizamos a análise exploratória e a extração de insights de negócio a partir dos Data Marts persistidos na **Camada Gold** (`copa_do_mundo.gold`).

---
**Estrutura de Análises:**
1. **Tópico 1:** Análise de Arbitragem
2. **Tópico 2:** Métricas de Seleções e Staff
3. **Tópico 3:** Análise de Partidas, controle de jogo e estratégia

In [0]:
%pip install seaborn

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F
import seaborn as sns

# Configuração de estilo global para os gráficos de visualização
sns.set_theme(style="whitegrid", palette="Blues_r")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

# Mapeamento do Catálogo e Schemas
CATALOG_NAME = "copa_do_mundo"
GOLD_SCHEMA = "gold"

print("🔄 Carregando tabelas da Camada Gold...")

# Leitura centralizada dos Data Marts da Gold
df_gold_squads = spark.read.table(
    f"{CATALOG_NAME}.{GOLD_SCHEMA}.gold_squad_metrics"
)
df_gold_players = spark.read.table(
    f"{CATALOG_NAME}.{GOLD_SCHEMA}.gold_player_performance"
)
df_gold_referees = spark.read.table(
    f"{CATALOG_NAME}.{GOLD_SCHEMA}.gold_referee_analytics"
)
df_gold_matches = spark.read.table(
    f"{CATALOG_NAME}.{GOLD_SCHEMA}.gold_match_summary"
)

print("✅ Tabelas da Camada Gold carregadas com sucesso com a instrução spark.read.table!")

**Tópico 1:** Análise de Arbitragem e Disciplina

**Atribuição por Confederação:**

**1.1** - Temos árbitros de diferentes Confederações escolhidos para a Copa. Qual a distribuição inicial por Confederação dos árbitros escolhidos? 

Para responder a esta pergunta excutamos a consulta diretamente na tabela *gold_referee_analytics*

Código Python:

In [0]:
from pyspark.sql import functions as F

# Leitura da tabela Gold de arbitragem
df_referees_gold = spark.read.table("copa_do_mundo.gold.gold_referee_analytics")

# Agregação por Confederação
df_confed_dist = (
    df_referees_gold.groupBy("confederacao")
    .agg(F.count("referee_id").alias("total_arbitros"))
    .orderBy(F.col("total_arbitros").desc())
)

display(df_confed_dist)

Graficamente:

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Converter o resultado para Pandas para plotagem local
pdf_confed = df_confed_dist.toPandas()

# Configuração do estilo visual
plt.figure(figsize=(10, 5))
sns.set_theme(style="whitegrid")

# Gráfico de barras
ax = sns.barplot(
    data=pdf_confed,
    x="confederacao",
    y="total_arbitros",
    palette="Blues_r"
)

# Rótulos e Título
plt.title("Distribuição Inicial de Árbitros por Confederação — Copa 2026", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Confederação", fontsize=12)
plt.ylabel("Total de Árbitros", fontsize=12)

# Adicionar o valor numérico no topo de cada barra
for p in ax.patches:
    height = int(p.get_height())
    if height > 0:
        ax.annotate(
            f"{height}",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
            xytext=(0, 3),
            textcoords="offset points",
        )

plt.tight_layout()
plt.show()

Alternativamente via consulta SQL:

In [0]:
%sql
SELECT 
    confederacao,
    COUNT(referee_id) AS total_arbitros
FROM copa_do_mundo.gold.gold_referee_analytics
GROUP BY confederacao
ORDER BY total_arbitros DESC;

In [0]:
%sql
SELECT *
FROM `copa_do_mundo`.`silver`.`referees_silver`

Distribuição (%):

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

# 1. Agregação por Confederação
df_confed_dist = (
    df_gold_referees.groupBy("confederacao")
    .agg(F.count("referee_id").alias("total_arbitros"))
    .orderBy(F.col("total_arbitros").desc())
)

# 2. Converte para Pandas e faz pivot para estrutura empilhada
pdf_confed = df_confed_dist.toPandas()
pdf_pivot = pdf_confed.set_index("confederacao").T

# 3. Criação do Gráfico de Barra Empilhada Vertical
fig, ax = plt.subplots(figsize=(6, 8))

# Paleta de cores para destacar as confederações
colors = [
    "#1f77b4",
    "#aec7e8",
    "#ff7f0e",
    "#ffbb78",
    "#2ca02c",
    "#98df8a",
    "#d62728",
]

# Gráfico de barra empilhada vertical (kind='bar')
pdf_pivot.plot(
    kind="bar", stacked=True, ax=ax, color=colors[: len(pdf_confed)], width=0.45
)

# 4. Rótulos de Valores e Porcentagens centralizados em cada segmento vertical
total_geral = pdf_confed["total_arbitros"].sum()
bottom_pos = 0

for idx, row in pdf_confed.iterrows():
    val = row["total_arbitros"]
    pct = (val / total_geral) * 100

    # Insere o texto apenas se o segmento for suficientemente grande (> 3%)
    if pct > 3:
        ax.text(
            0,
            bottom_pos + val / 2,
            f"{val}\n({pct:.1f}%)",
            ha="center",
            va="center",
            color="white",
            fontweight="bold",
            fontsize=10,
        )
    bottom_pos += val

# 5. Formatação Visual, Eixos e Legenda
plt.title(
    "Distribuição Proporcional de Árbitros\npor Confederação — Copa 2026",
    fontsize=13,
    fontweight="bold",
    pad=15,
)
ax.set_xticks([])  # Remove o eixo X (já que temos apenas uma barra única)
ax.set_ylabel("Quantidade Total de Árbitros", fontsize=11)
ax.set_ylim(0, total_geral * 1.02)  # Define limite vertical com margem

# Legenda posicionada ao lado da barra
plt.legend(
    title="Confederação",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    frameon=True,
)

plt.tight_layout()
plt.show()

**Tópico 1:** Análise de Arbitragem e Disciplina

**Atribuição por Confederação:**

**1.2** - Alguma Confederação atuou em mais jogos do que outras em todas as fases da Copa?

Para responder a essa pergunta separamos as fases em: Fase de Grupo, 16-avos de final, Fase Final (oitavas-de-final, quartas-de-final, semi-final, disputa de 3º colocado e final). A resposta foi gerada com base nas tabelas *matches_detailed_silver* e *gold_referee_analytics*. 

Resposta e Análise dos Resultados
1. A UEFA e a CONMEBOL mantêm o maior volume absoluto de jogos apitados na Fase de Grupos, refletindo a maior proporção de
árbitros convocados inicialmente.
2. Nas Fases Finais (Oitavas até a Final), há uma filtragem onde árbitros de confederações cujas seleções avançam para as fases
decisivas deixam de apitar (devido à neutralidade de confederação/país), elevando a presença relativa de árbitros neutros das
confederações com menor número de seleções remanescentes (como CONCACAF, AFC e CAF).

In [0]:
from pyspark.sql import functions as F

# 1. Leitura das tabelas necessárias
df_matches_silver = spark.read.table("copa_do_mundo.silver.matches_detailed_silver")
df_referees_gold = spark.read.table("copa_do_mundo.gold.gold_referee_analytics")

# 2. Mapeamento das 3 Fases Agrupadas
df_matches_staged = df_matches_silver.withColumn(
    "fase_agrupada",
    F.when(F.col("stage_name") == "Group Stage", "1. Fase de Grupos")
     .when(F.col("stage_name") == "Round of 32", "2. 16 avos de Final")
     .when(F.col("stage_name").isin("Round of 16", "Quarter-finals", "Semi-finals", "Third-place match", "Final"), "3. Fases Finais (Oitavas até Final)")
     .otherwise("Outros")
)

# 3. Cruzamento com os Árbitros e Agregação de Jogos Apitados por Confederação
df_confed_fases = (
    df_matches_staged.join(
        df_referees_gold.select("referee_name", "confederacao"),
        on="referee_name",
        how="inner"
    )
    .groupBy("confederacao", "fase_agrupada")
    .agg(F.countDistinct("match_id").alias("jogos_apitados"))
    .orderBy("fase_agrupada", F.col("jogos_apitados").desc())
)

display(df_confed_fases)

Para responder visualmente se alguma Confederação atuou em mais jogos do que outras em todas as fases, foi construído um gráfico de barras empilhadas agrupado por fase:

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Converter para Pandas para pivotagem e gráfico
pdf_fases = df_confed_fases.toPandas()

# Pivot para transformar a Confederação nas colunas do gráfico
pdf_pivot_fases = pdf_fases.pivot(
    index="fase_agrupada", 
    columns="confederacao", 
    values="jogos_apitados"
).fillna(0)

# Criar o gráfico
fig, ax = plt.subplots(figsize=(12, 6))

pdf_pivot_fases.plot(
    kind="bar",
    stacked=False, # Barras lado a lado para comparar diretamente entre confederações por fase
    ax=ax,
    colormap="tab10",
    width=0.8
)

# Formatação e Rótulos
plt.title("Atuação de Árbitros por Confederação em cada Fase da Copa do Mundo", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Fase da Competição", fontsize=12)
plt.ylabel("Quantidade de Jogos Apitados", fontsize=12)
plt.xticks(rotation=0, fontsize=10)

# Inserir valores numéricos no topo de cada barra
for p in ax.patches:
    height = int(p.get_height())
    if height > 0:
        ax.annotate(
            f"{height}",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold",
            xytext=(0, 2),
            textcoords="offset points"
        )

plt.legend(title="Confederação", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

Para responder visualmente se alguma Confederação atuou em mais jogos do que outras em todas as fases, foi construído um gráfico de barras empilhadas agrupado por fase:

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

# 1. Leitura e Mapeamento das 3 Fases Agrupadas
df_matches_silver = spark.read.table("copa_do_mundo.silver.matches_detailed_silver")
df_referees_gold = spark.read.table("copa_do_mundo.gold.gold_referee_analytics")

df_matches_staged = df_matches_silver.withColumn(
    "fase_agrupada",
    F.when(F.col("stage_name") == "Group Stage", "Fase de Grupos")
    .when(F.col("stage_name") == "Round of 32", "16-avos de Final")
    .when(F.col("stage_name").isin("Round of 16", "Quarter-finals", "Semi-finals", "Third-place match", "Final"), "Fases Finais")
    .otherwise("Outros"),
)

# 2. Agregação do Total de Jogos por Confederação em cada Fase
df_confed_fases = (
    df_matches_staged.join(
        df_referees_gold.select("referee_name", "confederacao"),
        on="referee_name",
        how="inner",
    )
    .groupBy("fase_agrupada", "confederacao")
    .agg(F.countDistinct("match_id").alias("jogos_apitados"))
)

# 3. Preparação dos dados em Pandas e ordenação lógica cronológica no eixo X
pdf_fases = df_confed_fases.toPandas()

ordem_fases = ["Fase de Grupos", "16-avos de Final", "Fases Finais"]
pdf_pivot = pdf_fases.pivot(
    index="fase_agrupada", columns="confederacao", values="jogos_apitados"
).fillna(0)
pdf_pivot = pdf_pivot.reindex(ordem_fases)

# 4. Construção do Gráfico com 3 Barras Empilhadas Verticais
fig, ax = plt.subplots(figsize=(10, 7))

# Paleta de cores para destacar as confederações
colors = [
    "#1f77b4",
    "#aec7e8",
    "#ff7f0e",
    "#ffbb78",
    "#2ca02c",
    "#98df8a",
    "#d62728",
]

pdf_pivot.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=colors[: len(pdf_pivot.columns)],
    width=0.45,
)

# 5. Adição dos rótulos numéricos e percentuais dentro de cada segmento vertical
for i, fase in enumerate(ordem_fases):
    row = pdf_pivot.loc[fase]
    total_fase = row.sum()
    bottom_pos = 0

    for confederacao, val in row.items():
        if val > 0:
            pct = (val / total_fase) * 100
            # Exibe o texto dentro do segmento apenas se for representativo (> 3.5%)
            if pct > 3.5:
                ax.text(
                    i,
                    bottom_pos + val / 2,
                    f"{int(val)}\n({pct:.1f}%)",
                    ha="center",
                    va="center",
                    color="white",
                    fontweight="bold",
                    fontsize=8.5,
                )
            bottom_pos += val

# 6. Formatação Visual, Eixos e Legenda
plt.title(
    "Distribuição da Arbitragem por Confederação entre as Fases da Copa 2026",
    fontsize=13,
    fontweight="bold",
    pad=20,
)
ax.set_xlabel("Fase da Copa", fontsize=11, fontweight="bold")
ax.set_ylabel("Quantidade Total de Jogos Apitados", fontsize=11)
plt.xticks(rotation=0, fontsize=10)

# Legenda externa mapeando as confederações por cor
plt.legend(
    title="Confederação",
    bbox_to_anchor=(1.03, 1),
    loc="upper left",
    frameon=True,
)

plt.tight_layout()
plt.show()

**Tópico 1:** Análise de Arbitragem e Disciplina

**Neutralidade de Arbitragem:**

**1.3**- Existem partidas em que a confederação do árbitro principal é idêntica à de uma das seleções participantes?

Historicamente, a FIFA proíbe estritamente que um árbitro apite jogos de sua própria seleção nacional. No entanto, para partidas envolvendo seleções da mesma confederação (por exemplo, um árbitro da UEFA apitando um jogo entre duas seleções europeias ou entre uma europeia e uma de outra confederação), as regras permitem em cenários específicos — especialmente em confrontos diretos entre equipes da mesma confederação ou nas fases finais do torneio.

Mapeamento Lógico e Cruzamentos Necessários
Para responder a essa pergunta e construir a tabela detalhada foi consultada a tabela *match_prediction_features_silver* filtrando eventos nos quais as Confederações do árbitro e de pelo menos 1 seleção são iguais.

Resposta:
Do total de 104 partidas na Copa, 39 partidas tiveram o árbitro e pelo menos 1 das seleções pertencentes à mesma Confederação de futebol.

In [0]:
from pyspark.sql import functions as F

# 1. Carregar tabela com dados de partidas, confederações e ranking FIFA
df_matches = spark.read.table("copa_do_mundo.silver.match_prediction_features_silver")

# 2. Filtrar partidas onde a Confederação do Árbitro é IGUAL à de ao menos uma das seleções
df_same_confed_matches = df_matches.filter(
    (F.col("referee_confederacao") == F.col("home_confederation")) |
    (F.col("referee_confederacao") == F.col("away_confederation"))
)

# 6. Selecionar e ordenar as colunas da tabela de resultado
tabela_jogos_mesma_confed = df_same_confed_matches.select(
    F.col("match_id"),
    F.col("stage_name").alias("fase"),
    F.col("home_team_name").alias("selecao_mandante"),
    F.col("home_confederation").alias("confed_mandante"),
    F.col("home_fifa_rank").alias("ranking_fifa_mandante"),
    F.col("away_team_name").alias("selecao_visitante"),
    F.col("away_confederation").alias("confed_visitante"),
    F.col("away_fifa_rank").alias("ranking_fifa_visitante"),
    F.col("referee_name").alias("arbitro"),
    F.col("referee_confederacao").alias("confed_arbitro")
).orderBy("match_id")

# Exibir a tabela detalhada
display(tabela_jogos_mesma_confed)

**Tópico 1:** Análise de Arbitragem e Disciplina

**Rigor Disciplinar:**

**1.4**- Quais são os 3 árbitros mais rigorosos da competição? Qual métrica pode ser usada para esta avaliação?

Para definir e classificar os árbitros mais rigorosos de forma robusta e matematicamente justa, não foram avaliados exclusivamente o volume absoluto de faltas apitadas, nem exclusivamente o histórico de cartões. 
A proposta para avaliação foi a criação do **Índice Composto de Rigor da Arbitragem** *indice_rigor* que avalia a média de faltas e a média de cartões. O objetivo da criação do índice foi evitar distorção por amostras (árbitros com poucos jogos apitados), e a intensidade da punição (um árbitro que apita muitas faltas simples é menos rigoroso do que um que expulsa um atelta por uma única falta).

Os dados foram filtrados para não considerar os árbitros com apenas 1 jogo apitado.

A tabela utilizada foi *referees_silver*.

Resposta: Os três árbitros mais rigorosos de acordo com o critério adota são: Yael Falcón (Argentina), Ma Ning (China) e Jalal Jayed (Marrocos).


In [0]:
from pyspark.sql import functions as F

# 1. Carregar os dados de arbitragem da Camada Gold / Silver
df_referees = spark.read.table("copa_do_mundo.silver.referees_silver")

# 2. Cálculo das Médias por Jogo e Construção do Índice Composto de Rigor
df_rigor_arbitros = (
    df_referees.filter(F.col("total_jogos_apitados") >= 2) # Filtro de relevância estatística
    .withColumn("media_faltas", F.round(F.col("total_faltas_marcadas") / F.col("total_jogos_apitados"), 2))
    .withColumn("media_cartoes", F.round(F.col("avg_cards_per_game"), 2))
    # Índice Ponderado de Rigor Disciplinar
    .withColumn(
        "indice_rigor",
        F.round(
            (F.col("media_faltas") * 1.0) + 
            (F.col("media_cartoes") * 3.0), 
            2
        )
    )
    .select(
        "referee_id",
        F.col("name").alias("referee_name"),
        "country",
        "confederacao",
        "total_jogos_apitados",
        "media_faltas",
        "media_cartoes",
        "indice_rigor"
    )
    .orderBy(F.col("indice_rigor").desc())
)

# 3. Exibir os Top 3 Árbitros mais rigorosos
top_3_rigorosos = df_rigor_arbitros.limit(3)
display(top_3_rigorosos)

**Tópico 2:** Seleções e Staff (Squad Metrics)

**Maturidade e Idade:**

**2.1**- Qual é a seleção com a maior e a menor média de idade do torneio? Existe relação entre a idade média da seleção e seu desempenho final?

A resposta pode ser obtida via consulta SQL.

As seleções com maiores médias de idade são: Panama (30), Irã (29,6), Colombia (29,6), Cabo verde (29,2) e Qatar (28,9).

As seleções com menores médias de idade são: Costa do Marfim (25,3), Ecuador (25,6), Bosnia e Herzegovina(26), Marrocos (26,1) e Espanha (26,2).

In [0]:
%sql
-- Seleção com maior e menor média de idade
WITH team_names AS (
    SELECT DISTINCT home_team_id AS team_id, home_team_name AS selecao
    FROM copa_do_mundo.silver.match_prediction_features_silver
    UNION
    SELECT DISTINCT away_team_id AS team_id, away_team_name AS selecao
    FROM copa_do_mundo.silver.match_prediction_features_silver
)
SELECT 
    tn.selecao,
    gsm.idade_media AS media_idade
FROM copa_do_mundo.gold.gold_squad_metrics gsm
JOIN team_names tn ON gsm.team_id = tn.team_id
ORDER BY media_idade DESC;

**Tópico 2:** Seleções e Staff (Squad Metrics)

**Maturidade e Idade:**

**2.1**- Relação entre Idade Média e Desempenho Final

Para responder se existe relação entre a idade e o desempenho, analisamos a correlação estatística (ou o cruzamento entre a média de idade e a fase de eliminação/posição final)

Entre as 4 primeiras seleções da Copa temos idades médias variando de 26,2 - Espanha até 28,7 - Argentina.

In [0]:
%sql
-- Seleção com maior e menor média de idade vs. desempenho final
WITH team_names AS (
    SELECT DISTINCT home_team_id AS team_id, home_team_name AS selecao
    FROM copa_do_mundo.silver.match_prediction_features_silver
    UNION
    SELECT DISTINCT away_team_id AS team_id, away_team_name AS selecao
    FROM copa_do_mundo.silver.match_prediction_features_silver
),
team_matches AS (
    -- Jogos como mandante
    SELECT
        home_team_id AS team_id,
        home_score AS gols_pro,
        away_score AS gols_contra,
        CASE
            WHEN stage_name = 'Group Stage' THEN 1
            WHEN stage_name = 'Round of 32' THEN 2
            WHEN stage_name = 'Round of 16' THEN 3
            WHEN stage_name = 'Quarter-finals' THEN 4
            WHEN stage_name = 'Semi-finals' THEN 5
            WHEN stage_name = 'Third-place match' THEN 6
            WHEN stage_name = 'Final' THEN 7
        END AS stage_order,
        CASE WHEN home_score > away_score THEN 1 ELSE 0 END AS vitoria
    FROM copa_do_mundo.silver.match_prediction_features_silver
    UNION ALL
    -- Jogos como visitante
    SELECT
        away_team_id AS team_id,
        away_score AS gols_pro,
        home_score AS gols_contra,
        CASE
            WHEN stage_name = 'Group Stage' THEN 1
            WHEN stage_name = 'Round of 32' THEN 2
            WHEN stage_name = 'Round of 16' THEN 3
            WHEN stage_name = 'Quarter-finals' THEN 4
            WHEN stage_name = 'Semi-finals' THEN 5
            WHEN stage_name = 'Third-place match' THEN 6
            WHEN stage_name = 'Final' THEN 7
        END AS stage_order,
        CASE WHEN away_score > home_score THEN 1 ELSE 0 END AS vitoria
    FROM copa_do_mundo.silver.match_prediction_features_silver
),
team_performance AS (
    SELECT
        team_id,
        SUM(vitoria) AS vitorias,
        SUM(gols_pro) AS gols_pro,
        SUM(gols_contra) AS gols_contra,
        MAX(stage_order) AS fase_max
    FROM team_matches
    GROUP BY team_id
)
SELECT
    tn.selecao,
    gsm.idade_media AS media_idade,
    CASE tp.fase_max
        WHEN 1 THEN 'Group Stage'
        WHEN 2 THEN 'Round of 32'
        WHEN 3 THEN 'Round of 16'
        WHEN 4 THEN 'Quarter-finals'
        WHEN 5 THEN 'Semi-finals'
        WHEN 6 THEN 'Third-place match'
        WHEN 7 THEN 'Final'
    END AS fase_final_alcancada,
    tp.vitorias,
    tp.gols_pro - tp.gols_contra AS saldo_gols
FROM copa_do_mundo.gold.gold_squad_metrics gsm
JOIN team_names tn ON gsm.team_id = tn.team_id
JOIN team_performance tp ON gsm.team_id = tp.team_id
ORDER BY tp.fase_max DESC, media_idade DESC;

**Tópico 2:** Seleções e Staff (Squad Metrics)

**Maturidade e Idade:**

**2.2**- Qual a média de idade por posição (Goleiro - GK, Defensor - DEF, Meio-campo - MID, Atacante - FWD) e qual setor permite atletas mais veteranos?

Para responder a essa pergunta utilizamos a tabela *squads_and_players_silver*. As estatisticas de idade por posição são também apresentadas graficamente para facilitar a análise e comparação.

A posição de goleiro permite atletas mais veteranos pelos dados analisados. É provável que isso aconteça por 2 motivos: menor exigência de intensidade física em comparação com as demais posições, e a escolha de atletas mais experientes e maduros para uma posição tão importante como goleiro. 

In [0]:
from pyspark.sql import functions as F

# 1. Leitura da tabela Silver de jogadores e convocados
df_players = spark.read.table("copa_do_mundo.silver.squads_and_players_silver")

# 1.1 Cálculo da idade a partir da data de nascimento
df_players = df_players.withColumn(
    "age",
    F.floor(F.datediff(F.current_date(), F.col("date_of_birth")) / 365.25)
)

# 2. Agregação de idade por posição padronizada
df_idade_posicao = (
    df_players.groupBy("position")  # ou "posicao_agrupada" / "position_category"
    .agg(
        F.round(F.avg("age"), 2).alias("media_idade"),
        F.max("age").alias("idade_maxima"),
        F.min("age").alias("idade_minima"),
        F.count("player_id").alias("total_jogadores"),
    )
    .orderBy(F.col("media_idade").desc())
)

display(df_idade_posicao)

Analise grafica:

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import functions as F

# 1. Leitura e cálculo da idade dos jogadores
DATA_INICIO_COPA = F.lit("2026-06-11").cast("date")

df_players = spark.read.table("copa_do_mundo.silver.squads_and_players_silver")

df_players_age = df_players.withColumn(
    "age",
    F.floor(F.datediff(DATA_INICIO_COPA, F.col("date_of_birth")) / 365.25),
)

# 2. Converter para Pandas apenas os campos necessários para o gráfico
pdf_players = df_players_age.select("position", "age").toPandas()

# Definir a ordem exata das posições no gráfico
ordem_posicoes = ["GK", "DEF", "MID", "FWD"]

# 3. Configuração do Estilo e Figura
plt.figure(figsize=(11, 6))
sns.set_theme(style="whitegrid")

# 4. Geração do Boxplot com Indicação de Média, Mediana e Outliers
ax = sns.boxplot(
    data=pdf_players,
    x="position",
    y="age",
    order=ordem_posicoes,
    palette="Blues",
    showmeans=True,  # Exibe a média como um ponto/marcador
    meanprops={
        "marker": "o",
        "markerfacecolor": "red",
        "markeredgecolor": "red",
        "markersize": "7",
    },
    flierprops={
        "marker": "o",
        "markerfacecolor": "orange",
        "markeredgecolor": "black",
        "markersize": "6",
    },  # Outliers
    boxprops={"alpha": 0.8},
)

# 5. Adicionar anotações explícitas de média e mediana por posição
for i, pos in enumerate(ordem_posicoes):
    ages = pdf_players[pdf_players["position"] == pos]["age"].dropna()
    if len(ages) > 0:
        media = ages.mean()
        mediana = ages.median()

        # Inserir texto formatado acima de cada Box
        ax.text(
            i,
            pdf_players["age"].max() + 0.8,
            f"Média: {media:.1f}\nMediana: {mediana:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
            fontweight="bold",
            bbox=dict(
                boxstyle="round,pad=0.3",
                facecolor="white",
                edgecolor="gray",
                alpha=0.8,
            ),
        )

# 6. Formatação Visual e Legendas
plt.title(
    "Distribuição e Variabilidade de Idade dos Jogadores por Posição na Copa 2026",
    fontsize=14,
    fontweight="bold",
    pad=25,
)
plt.xlabel("Posição", fontsize=12, fontweight="bold")
plt.ylabel("Idade (Anos)", fontsize=12, fontweight="bold")
plt.ylim(
    pdf_players["age"].min() - 2, pdf_players["age"].max() + 5
)  # Espaço para rótulos superiores

# Legenda personalizada para Média e Outlier
from matplotlib.lines import Line2D

legend_elements = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label="Média",
        markerfacecolor="red",
        markersize=8,
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label="Mediana (Linha Central)",
        markerfacecolor="gray",
        markersize=1,
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        label="Outlier",
        markerfacecolor="orange",
        markeredgecolor="black",
        markersize=7,
    ),
]
ax.legend(handles=legend_elements, loc="lower right", frameon=True)

plt.tight_layout()
plt.show()

**Tópico 2:** Seleções e Staff (Squad Metrics)

**Valor de Mercado:**

**2.3**- Quais são as 5 seleções mais valiosas em valor total de mercado e em valor médio por atleta?

Para analisar o Valor de Mercado na camada Gold (gold_squad_metrics), consideram-se duas abordagens complementares: o Valor Total do Elenco (soma do valor de mercado de todos os atletas convocados) e o Valor Médio por Atleta (valor total dividido pelo número de atletas na convocação oficial - 26 jogadores).

TOP 5 em Valor Total de Mercado e Valor médio por atleta:
Espanha, França, Inglaterra, Portugal e Argentina.


In [0]:
from pyspark.sql import functions as F

# 1. Leitura da tabela Gold de métricas de plantel/convocações
df_squad_metrics = spark.read.table("copa_do_mundo.gold.gold_squad_metrics")

# 1.1 Mapeamento de nomes de seleções e confederações
df_team_info = (
    spark.read.table("copa_do_mundo.silver.match_prediction_features_silver")
    .select(
        F.col("home_team_id").alias("team_id"),
        F.col("home_team_name").alias("selecao"),
        F.col("home_confederation").alias("confederacao"),
    )
    .union(
        spark.read.table("copa_do_mundo.silver.match_prediction_features_silver")
        .select(
            F.col("away_team_id").alias("team_id"),
            F.col("away_team_name").alias("selecao"),
            F.col("away_confederation").alias("confederacao"),
        )
    )
    .distinct()
)
df_squad_metrics = df_squad_metrics.join(df_team_info, on="team_id", how="inner")

# 2. Ranking 1: Top 5 Seleções por Valor Total de Mercado
top5_valor_total = (
    df_squad_metrics.select(
        "selecao",
        "confederacao",
        F.col("total_convocados").alias("total_jogadores"),
        "valor_total_mercado_eur",
        F.col("valor_medio_jogador_eur").alias("valor_medio_atleta_eur"),
    )
    .orderBy(F.col("valor_total_mercado_eur").desc())
    .limit(5)
)

print("=== TOP 5 SELEÇÕES MAIS VALIOSAS (VALOR TOTAL) ===")
display(top5_valor_total)

# 3. Ranking 2: Top 5 Seleções por Valor Médio por Atleta
top5_valor_medio = (
    df_squad_metrics.select(
        "selecao",
        "confederacao",
        F.col("total_convocados").alias("total_jogadores"),
        "valor_total_mercado_eur",
        F.col("valor_medio_jogador_eur").alias("valor_medio_atleta_eur"),
    )
    .orderBy(F.col("valor_medio_atleta_eur").desc())
    .limit(5)
)

print("=== TOP 5 SELEÇÕES MAIS VALIOSAS (MÉDIA POR ATLETA) ===")
display(top5_valor_medio)

**Tópico 2:** Seleções e Staff (Squad Metrics)

**Diversidade de Clubes:**

**2.4**- Qual seleção possui a maior pulverização de atletas em diferentes clubes de origem?

Para responder a essa pergunta foi analisada na camada Gold a tabela *gold_squad_metrics*. Foi criado o índice de pulverização *indice_pulverizacao* (total de clubes representados em cada seleção / total de convocados). Um índice de valor "1" indica que os 26 convocados de uma seleção jogam em clubes distintos.

O resultado é apresentado na tabela final.

Analisando os dados podemos observar que:
- Cabo Verde e Suécia possuem índice "1", ou seja, todos os 26 convocados de cada seleção jogam em clubes distintos.
- A campeã Espanha possui índice "0,5", ou seja, os 26 jogadores convocados estão distribuídos em apenas 13 clubes. Analisando os dados da Espanha temos: Barcelona (8 convocados), Atlético de Madrid (3), Athletic Bilbao (3) e Arsenal (3).

In [0]:
from pyspark.sql import functions as F

# Leitura da tabela Gold de métricas de plantel
df_squad_metrics = spark.read.table("copa_do_mundo.gold.gold_squad_metrics")

# Mapeamento de nomes de seleções e confederações
df_team_info = (
    spark.read.table("copa_do_mundo.silver.match_prediction_features_silver")
    .select(
        F.col("home_team_id").alias("team_id"),
        F.col("home_team_name").alias("selecao"),
        F.col("home_confederation").alias("confederacao"),
    )
    .union(
        spark.read.table("copa_do_mundo.silver.match_prediction_features_silver")
        .select(
            F.col("away_team_id").alias("team_id"),
            F.col("away_team_name").alias("selecao"),
            F.col("away_confederation").alias("confederacao"),
        )
    )
    .distinct()
)
df_squad_metrics = df_squad_metrics.join(df_team_info, on="team_id", how="inner")

# Seleções com maior total de clubes representados
df_club_diversity = (
    df_squad_metrics.select(
        "selecao",
        "confederacao",
        F.col("total_convocados").alias("total_jogadores"),
        "total_clubes_representados",
        F.round(
            F.col("total_clubes_representados") / F.col("total_convocados"), 2
        ).alias("indice_pulverizacao"),
    )
    .orderBy(F.col("total_clubes_representados").desc())
    .limit(48)
)

display(df_club_diversity)

**Tópico 3:** Análise de Partidas, controle de jogo e estratégia

*Em construção:*
